# 00｜IMDB v1.1 去重版資料準備

**依批准的 v1.1 規格執行。** 不覆蓋舊 v1.0。完整清理後文字 SHA-256 作為重複判定；官方 Test 25,000 筆完全保留（包含其內部重複）。

資料流程：來源驗證 → 官方 Test 保留 → 排除 Train 中與 Test 相同文本 123 筆 → 訓練池內去重 98 筆 → seed 42 分層 80/20 切分成 19,823 / 4,956。

In [1]:
from pathlib import Path
import json
import sys
ROOT=Path('/workspace/imdb_matrix_experiment/variants/dedup_v1_1')
assert ROOT.exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from src.data_utils import prepare
from src.common import config,file_hash,load_splits
import pandas as pd
config_obj, config_sha=config()
print("Variant:",config_obj["version"])
print("Config SHA256:",config_sha)
for name in ("train","test"):
    source=Path(config_obj["dataset"][f"source_{name}"])
    actual=file_hash(source)
    assert actual==config_obj["dataset"][f"sha256_{name}"]
    print("Source:",name,source.stat().st_size,actual)


Variant: dedup_v1.1
Config SHA256: 6483ed250a1ddda6e54da5f6120e41180c8b0215bc90cf4ac68f71c874b64664
Source: train 20979968 db47d16b5c297cc0dd625e519c81319c24c9149e70e8496de5475f6fa928342c
Source: test 20470363 b52e26e2f872d282ffac460bf9770b25ac6f102cda0e6ca7158df98c94e8b3da


## 1. 重建獨立 v1.1 切分與稽核清單
系統會拒絕已變更的來源 SHA 或與批准數量及 ID 雜湊不同的資料切分。 Train/Validation/官方 Test 的原始資料與清理後完整文字由專案內 manifest 追蹤。

In [2]:
manifest=prepare()
print("Counts:")
print(pd.DataFrame(manifest["counts"]).T.to_string())
print("Exclusions:",manifest["audit"]["excluded_test_overlap"],manifest["audit"]["excluded_intra_train_duplicates"])
print("Split SHA:",manifest["split_assignments_sha256"])


Counts:
             rows                    labels                                                            sha256                                                 source_ids_sha256 unique_cleaned_text
train       19823    {'0': 9869, '1': 9954}  119f5201290bc5d555354738ec7bd2410d3f876508058ee393966227c9bb62ad  0ac4e54e9f9dea0292d10a8349cfb554e58456271ccddffa476d0510b494d0b2               19823
validation   4956    {'0': 2467, '1': 2489}  d04f164fd94352fc5156548e23e5a8d885053f599252d9b273eca1ff38cf7fc8  bf88a23488e4ccc54ef37262e66834981b9dd34c9ab7284bd7149f42995bfc9c                4956
test        25000  {'0': 12500, '1': 12500}  66ff71f324dc3e43f534e59a54ba2d0587ac6e44526cc21c41d58a291d839e34  933e8bd59bc47a8a9508418f628023403bd6c9c23823a25667e7b39322b2bb3c               24799
Exclusions: {'rows': 123, 'labels': {'0': 95, '1': 28}} {'rows': 98, 'labels': {'0': 69, '1': 29}}
Split SHA: 9e25b0d714d1fb555c8c8a58fa6858c392ec60ecfba554b923553626154174e9


## 2. 防洩漏檢查：不同 split 的完整清理後文本沒有完全重複
檢查的是完整清理後字串，不是 160-token 截斷序列。 並非語意相似度／近似重複檢查。

In [3]:
splits=load_splits()
hashes={n:set(df.cleaned_hash) for n,df in splits.items()}
assert not(hashes["train"]&hashes["validation"])
assert not(hashes["train"]&hashes["test"])
assert not(hashes["validation"]&hashes["test"])
assert len(splits["train"])==19823 and len(splits["validation"])==4956 and len(splits["test"])==25000
print("Cross-split identical cleaned text: 0 / 0 / 0")
for name,df in splits.items():
    print(name,len(df),df.label.value_counts().sort_index().to_dict())
print("Official Test retained:",manifest["audit"]["official_test_rows"])


Cross-split identical cleaned text: 0 / 0 / 0
train 19823 {0: 9869, 1: 9954}
validation 4956 {0: 2467, 1: 2489}
test 25000 {0: 12500, 1: 12500}
Official Test retained: 25000


## 3. 測試集限制與排除明細
官方 Test 自身的 201 筆額外重複仍予保留，避免破壞官方 Test 25k 基準。 排除資料僅儲存 ID、hash、label 與原因，不散佈評論全文。

In [4]:
audit=manifest["audit"]
print("Test internal duplicate groups:",audit["test_internal_duplicate_groups"])
print("Test extra duplicate rows:",audit["test_internal_extra_duplicate_rows"])
print("Caveat:",audit["warning"])
for name in ("excluded_test_overlap.csv","excluded_intra_train_duplicates.csv"):
    df=pd.read_csv(ROOT/"data/processed"/name)
    print(name,"rows",len(df),"labels",df.label.value_counts().sort_index().to_dict())


Test internal duplicate groups: 193
Test extra duplicate rows: 201
Caveat: No cross-split identical full cleaned text; semantic near-duplicates not tested and official Test internal duplicates retained.
excluded_test_overlap.csv rows 123 labels {0: 95, 1: 28}
excluded_intra_train_duplicates.csv rows 98 labels {0: 69, 1: 29}
